# Minimum SAE Circuit Discovery MVP

This Colab notebook implements the first experiment from the minimum SAE circuit discovery plan:

1. Load GPT-2-small with `HookedSAETransformer.from_pretrained_no_processing`.
2. Load the pretrained layer 8 residual-stream SAE from `gpt2-small-res-jb`.
3. Generate a self-contained IOI-style dataset.
4. Cache mean absolute SAE feature activations.
5. Run a magnitude top-K baseline.
6. Plot circuit size vs faithfulness.

The notebook is designed to live in GitHub and be opened from Colab. Results are written to Google Drive in versioned trial folders so failed or weak runs do not overwrite previous trials.


## 1. Colab setup

Run this cell first. Use a GPU runtime (`Runtime -> Change runtime type -> GPU`). The experiment is designed for Colab Pro+ A100 or V100, but the smoke test is small enough to verify the pipeline quickly.


In [ ]:
%pip install -q "sae-lens>=6,<7" pandas matplotlib tqdm


In [ ]:
import json
import os
import random
import shutil
from collections import defaultdict
from datetime import datetime
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from sae_lens import SAE, HookedSAETransformer
from tqdm.auto import tqdm

SEED = 12345
random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("This notebook expects a CUDA GPU. In Colab, enable Runtime -> Change runtime type -> GPU.")

device = "cuda"
torch.set_grad_enabled(False)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")


## 2. Output directory and run config

In Colab, outputs are saved to Google Drive under `MyDrive/minimum_sae_circuit_discovery`. Each run writes to `runs/<RUN_VERSION>/<TRIAL_ID>/`, shared caches live under `cache/<RUN_VERSION>/`, and the latest completed artifacts are mirrored to `latest/<RUN_VERSION>/`.


In [ ]:
MOUNT_DRIVE = True
PROJECT_DIR_NAME = "minimum_sae_circuit_discovery"
RUN_VERSION = "v001_mvp_magnitude"
TRIAL_ID = os.environ.get("TRIAL_ID") or datetime.now().strftime("trial_%Y%m%d_%H%M%S")
RUN_STARTED_AT = datetime.now().astimezone().isoformat(timespec="seconds")

try:
    from google.colab import drive  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_DIR_NAME
else:
    PROJECT_ROOT = Path.cwd() / f"{PROJECT_DIR_NAME}_outputs"

CACHE_DIR = PROJECT_ROOT / "cache" / RUN_VERSION
OUTPUT_DIR = PROJECT_ROOT / "runs" / RUN_VERSION / TRIAL_ID
LATEST_DIR = PROJECT_ROOT / "latest" / RUN_VERSION
for directory in (CACHE_DIR, OUTPUT_DIR, LATEST_DIR):
    directory.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "gpt2-small"
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.8.hook_resid_pre"
SAE_HOOK_NAME = "blocks.8.hook_resid_pre"
EXPECTED_D_SAE = 24_576

SMOKE_N_PROMPTS = 8
N_PROMPTS = 1_000
BATCH_SIZE = 16
K_VALUES = [5, 10, 20, 50, 100, 200, 500, 1_000]

CACHE_PATH = CACHE_DIR / "mean_abs_sae_features_layer8_ioi.pt"
SMOKE_CACHE_PATH = CACHE_DIR / "smoke_mean_abs_sae_features_layer8_ioi.pt"
RESULTS_CSV_PATH = OUTPUT_DIR / "magnitude_pareto_results.csv"
PLOT_PATH = OUTPUT_DIR / "magnitude_pareto.png"
SMOKE_RESULTS_CSV_PATH = OUTPUT_DIR / "smoke_magnitude_pareto_results.csv"
SMOKE_PLOT_PATH = OUTPUT_DIR / "smoke_magnitude_pareto.png"
MANIFEST_PATH = OUTPUT_DIR / "run_manifest.json"


def write_run_manifest(status, extra=None):
    manifest = {
        "status": status,
        "run_started_at": RUN_STARTED_AT,
        "run_updated_at": datetime.now().astimezone().isoformat(timespec="seconds"),
        "run_version": RUN_VERSION,
        "trial_id": TRIAL_ID,
        "seed": SEED,
        "model_name": MODEL_NAME,
        "sae_release": SAE_RELEASE,
        "sae_id": SAE_ID,
        "sae_hook_name": SAE_HOOK_NAME,
        "n_prompts": N_PROMPTS,
        "smoke_n_prompts": SMOKE_N_PROMPTS,
        "batch_size": BATCH_SIZE,
        "k_values": K_VALUES,
        "project_root": str(PROJECT_ROOT),
        "cache_dir": str(CACHE_DIR),
        "output_dir": str(OUTPUT_DIR),
        "latest_dir": str(LATEST_DIR),
        "artifacts": {
            "cache_path": str(CACHE_PATH),
            "results_csv_path": str(RESULTS_CSV_PATH),
            "plot_path": str(PLOT_PATH),
            "smoke_results_csv_path": str(SMOKE_RESULTS_CSV_PATH),
            "smoke_plot_path": str(SMOKE_PLOT_PATH),
        },
        "extra": extra or {},
    }
    with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)
    return manifest


def mirror_artifacts_to_latest(paths):
    LATEST_DIR.mkdir(parents=True, exist_ok=True)
    copied = []
    for artifact_path in paths:
        artifact_path = Path(artifact_path)
        if artifact_path.exists():
            destination = LATEST_DIR / artifact_path.name
            shutil.copy2(artifact_path, destination)
            copied.append(destination)
    return copied

write_run_manifest("initialized")

print(f"Google Drive project root: {PROJECT_ROOT}")
print(f"Run version: {RUN_VERSION}")
print(f"Trial ID: {TRIAL_ID}")
print(f"This run output folder: {OUTPUT_DIR}")
print(f"Shared cache folder: {CACHE_DIR}")
print(f"Latest completed artifacts folder: {LATEST_DIR}")


## 3. Load GPT-2-small and the layer 8 SAE

SAELens v6 returns the SAE object directly from `SAE.from_pretrained(...)`. The model is loaded with `from_pretrained_no_processing(...)` so the SAE sees raw activations matching its training setup.


In [ ]:
sae = SAE.from_pretrained(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device,
)
sae.eval()
for param in sae.parameters():
    param.requires_grad_(False)

metadata = getattr(sae.cfg, "metadata", None)
model_kwargs = getattr(metadata, "model_from_pretrained_kwargs", None) or {}
model = HookedSAETransformer.from_pretrained_no_processing(
    MODEL_NAME,
    device=device,
    **model_kwargs,
)
model.eval()

if getattr(model.tokenizer, "pad_token", None) is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"

print(f"Loaded model: {MODEL_NAME}")
print(f"Loaded SAE: {SAE_RELEASE} / {SAE_ID}")
print(f"SAE d_in: {sae.cfg.d_in}")
print(f"SAE d_sae: {sae.cfg.d_sae}")
assert sae.cfg.d_sae == EXPECTED_D_SAE, f"Expected d_sae={EXPECTED_D_SAE}, got {sae.cfg.d_sae}"


## 4. Generate a self-contained IOI-style dataset

Each row contains a clean prompt and a corrupt prompt. The MVP faithfulness metric uses the clean prompt and compares the logit for the correct indirect object against the logit for the incorrect subject at the final token position. The corrupt prompt is kept in the table for future activation-patching extensions.


In [ ]:
NAMES = [
    "John", "Mary", "Bob", "Alice", "Tom", "Sarah", "James", "Emily",
    "Robert", "Laura", "Michael", "Anna", "David", "Lisa", "Daniel", "Emma",
    "Paul", "Karen", "Mark", "Susan", "Peter", "Linda", "Kevin", "Nancy",
    "Steven", "Helen", "George", "Carol", "Brian", "Julia", "Henry", "Megan",
    "Adam", "Rachel", "Patrick", "Olivia", "Andrew", "Grace", "Edward", "Sophie",
]

PLACES = [
    "store", "park", "school", "office", "garden", "library", "station", "market",
    "museum", "theater", "church", "beach", "cafe", "hotel", "airport", "hospital",
]

OBJECTS = [
    "book", "letter", "drink", "snack", "ticket", "phone", "gift", "photo",
    "bag", "card", "key", "toy", "coin", "map", "note", "pen",
]

TEMPLATES = [
    "When {subject} and {io} went to the {place}, {subject} gave a {obj} to",
    "After {subject} and {io} visited the {place}, {subject} handed a {obj} to",
    "While {subject} and {io} waited near the {place}, {subject} passed a {obj} to",
    "Because {subject} and {io} were at the {place}, {subject} offered a {obj} to",
]


def build_ioi_dataset(n_prompts=1_000, seed=0):
    """Build a simple IOI-style dataset with clean and corrupt prompt pairs."""
    rng = random.Random(seed)
    records = []
    seen = set()
    attempts = 0

    while len(records) < n_prompts and attempts < n_prompts * 100:
        attempts += 1
        subject, io = rng.sample(NAMES, 2)
        place = rng.choice(PLACES)
        obj = rng.choice(OBJECTS)
        template = rng.choice(TEMPLATES)

        clean_prompt = template.format(subject=subject, io=io, place=place, obj=obj)
        corrupt_prompt = template.format(subject=io, io=subject, place=place, obj=obj)
        key = (clean_prompt, corrupt_prompt)
        if key in seen:
            continue
        seen.add(key)

        records.append(
            {
                "clean_prompt": clean_prompt,
                "corrupt_prompt": corrupt_prompt,
                "answer_clean": f" {io}",
                "answer_corrupt": f" {subject}",
                "subject": subject,
                "indirect_object": io,
                "place": place,
                "object": obj,
                "template": template,
            }
        )

    if len(records) < n_prompts:
        raise ValueError(f"Only generated {len(records)} unique prompts out of requested {n_prompts}.")

    return pd.DataFrame(records)


def token_id_for_answer(answer):
    tokens = model.to_tokens(answer, prepend_bos=False).reshape(-1)
    if tokens.numel() != 1:
        pieces = model.to_str_tokens(answer, prepend_bos=False)
        raise ValueError(f"Answer {answer!r} is not a single token: {pieces}")
    return int(tokens.item())


def get_answer_token_ids(row):
    """Return clean and corrupt answer token ids, validating one-token answers."""
    return token_id_for_answer(row["answer_clean"]), token_id_for_answer(row["answer_corrupt"])


def add_answer_token_ids(dataset):
    dataset = dataset.copy()
    clean_ids = []
    corrupt_ids = []
    for _, row in dataset.iterrows():
        clean_id, corrupt_id = get_answer_token_ids(row)
        clean_ids.append(clean_id)
        corrupt_ids.append(corrupt_id)
    dataset["answer_clean_id"] = clean_ids
    dataset["answer_corrupt_id"] = corrupt_ids
    return dataset

smoke_dataset = add_answer_token_ids(build_ioi_dataset(SMOKE_N_PROMPTS, seed=SEED))
full_dataset = add_answer_token_ids(build_ioi_dataset(N_PROMPTS, seed=SEED))

print(smoke_dataset[["clean_prompt", "answer_clean", "answer_corrupt"]].head())
print(f"Smoke prompts: {len(smoke_dataset)}")
print(f"Full prompts: {len(full_dataset)}")


## 5. Model, hook, and metric helpers

The full model is always evaluated without SAE substitution. A circuit run replaces the layer 8 residual stream with `sae.decode(mask * sae.encode(activation))`. This keeps SAE reconstruction error separate from circuit incompleteness.


In [ ]:
def group_tokenized_texts(texts):
    """Tokenize texts and group tensors by sequence length to avoid padding artifacts."""
    groups = defaultdict(list)
    for index, text in enumerate(texts):
        tokens = model.to_tokens(text, prepend_bos=True).squeeze(0).to(device)
        groups[int(tokens.numel())].append((index, tokens))
    return groups


def sae_mask_hook(activation, hook, mask=None):
    """Replace residual activations with an SAE reconstruction, optionally masking features."""
    feature_acts = sae.encode(activation)
    if mask is not None:
        mask = mask.to(device=feature_acts.device, dtype=feature_acts.dtype).view(1, 1, -1)
        feature_acts = feature_acts * mask
    return sae.decode(feature_acts)


def run_logits(texts, batch_size=BATCH_SIZE, mask=None, use_sae_substitution=False):
    """Return final-token logits for each text, preserving input order."""
    if isinstance(texts, str):
        texts = [texts]

    output_logits = [None] * len(texts)
    groups = group_tokenized_texts(texts)

    fwd_hooks = []
    if use_sae_substitution:
        fwd_hooks = [(SAE_HOOK_NAME, partial(sae_mask_hook, mask=mask))]

    with torch.no_grad():
        context = model.hooks(fwd_hooks=fwd_hooks) if fwd_hooks else torch.no_grad()
        with context:
            for items in groups.values():
                for start in range(0, len(items), batch_size):
                    chunk = items[start : start + batch_size]
                    indices = [item[0] for item in chunk]
                    tokens = torch.stack([item[1] for item in chunk], dim=0)
                    logits = model(tokens)
                    final_logits = logits[:, -1, :].detach().cpu()
                    for idx, row_logits in zip(indices, final_logits):
                        output_logits[idx] = row_logits

    return torch.stack(output_logits, dim=0)


def mean_logit_diff(final_logits, dataset):
    clean_ids = torch.tensor(dataset["answer_clean_id"].to_list(), dtype=torch.long)
    corrupt_ids = torch.tensor(dataset["answer_corrupt_id"].to_list(), dtype=torch.long)
    row_indices = torch.arange(len(dataset), dtype=torch.long)
    diffs = final_logits[row_indices, clean_ids] - final_logits[row_indices, corrupt_ids]
    return diffs.mean()


def compute_faithfulness(mask, dataset, full_logit_diff=None, batch_size=BATCH_SIZE):
    """Compute faithfulness = masked logit-diff / full-model logit-diff."""
    texts = dataset["clean_prompt"].to_list()

    if full_logit_diff is None:
        full_logits = run_logits(texts, batch_size=batch_size, use_sae_substitution=False)
        full_logit_diff = mean_logit_diff(full_logits, dataset)

    masked_logits = run_logits(
        texts,
        batch_size=batch_size,
        mask=mask,
        use_sae_substitution=True,
    )
    masked_logit_diff = mean_logit_diff(masked_logits, dataset)

    denominator = float(full_logit_diff.item())
    if abs(denominator) < 1e-8:
        raise ValueError(f"Full-model logit diff is too close to zero: {denominator}")

    faithfulness = float((masked_logit_diff / full_logit_diff).item())
    active_features = int(mask.sum().item()) if mask is not None else int(sae.cfg.d_sae)

    return {
        "faithfulness": faithfulness,
        "full_logit_diff": float(full_logit_diff.item()),
        "masked_logit_diff": float(masked_logit_diff.item()),
        "active_features": active_features,
    }


## 6. Dimension checks

This verifies that the SAE input dimension matches the layer 8 residual stream and that the SAE width is the expected 24,576 features.


In [ ]:
sample_text = smoke_dataset.loc[0, "clean_prompt"]
sample_tokens = model.to_tokens(sample_text, prepend_bos=True).to(device)
_, sample_cache = model.run_with_cache(sample_tokens, names_filter=[SAE_HOOK_NAME])
sample_acts = sample_cache[SAE_HOOK_NAME]

print(f"Sample activation shape at {SAE_HOOK_NAME}: {tuple(sample_acts.shape)}")
print(f"SAE d_in: {sae.cfg.d_in}")
print(f"SAE d_sae: {sae.cfg.d_sae}")

assert sample_acts.shape[-1] == sae.cfg.d_in, "SAE input dimension does not match the hooked activation dimension."
assert sae.cfg.d_sae == EXPECTED_D_SAE, f"Expected {EXPECTED_D_SAE} SAE features."


## 7. Cache mean absolute SAE feature activations

The magnitude baseline ranks features by their mean absolute activation over the clean prompts. The full-run cache is saved to Drive so reruns can skip this pass.


In [ ]:
def cache_mean_abs_sae_features(dataset, cache_path, batch_size=BATCH_SIZE, force_recompute=False):
    cache_path = Path(cache_path)
    if cache_path.exists() and not force_recompute:
        payload = torch.load(cache_path, map_location="cpu")
        mean_abs = payload["mean_abs"]
        print(f"Loaded cached feature magnitudes from {cache_path}")
        return mean_abs.to(device)

    running_sum = torch.zeros(int(sae.cfg.d_sae), device=device)
    total_prompts = 0
    texts = dataset["clean_prompt"].to_list()
    groups = group_tokenized_texts(texts)

    with torch.no_grad():
        for items in tqdm(groups.values(), desc="Token length groups"):
            for start in tqdm(range(0, len(items), batch_size), leave=False, desc="Batches"):
                chunk = items[start : start + batch_size]
                tokens = torch.stack([item[1] for item in chunk], dim=0)
                _, cache = model.run_with_cache(tokens, names_filter=[SAE_HOOK_NAME])
                acts = cache[SAE_HOOK_NAME]
                feature_acts = sae.encode(acts)
                per_prompt_mean = feature_acts.abs().mean(dim=1)
                running_sum += per_prompt_mean.sum(dim=0)
                total_prompts += per_prompt_mean.shape[0]

    mean_abs = (running_sum / total_prompts).detach().cpu()
    payload = {
        "mean_abs": mean_abs,
        "n_prompts": int(total_prompts),
        "hook_name": SAE_HOOK_NAME,
        "sae_release": SAE_RELEASE,
        "sae_id": SAE_ID,
    }
    torch.save(payload, cache_path)
    print(f"Saved feature magnitude cache to {cache_path}")
    return mean_abs.to(device)


def make_topk_mask(top_features, k):
    mask = torch.zeros(int(sae.cfg.d_sae), device=device)
    mask[top_features[:k].to(device)] = 1.0
    return mask


## 8. Magnitude sweep and plotting helpers

`run_magnitude_sweep` computes the full-model denominator once, evaluates an all-feature SAE reconstruction baseline once, and then sweeps top-K masks.


In [ ]:
def run_magnitude_sweep(k_values, dataset, top_features, batch_size=BATCH_SIZE, output_csv_path=None):
    texts = dataset["clean_prompt"].to_list()
    full_logits = run_logits(texts, batch_size=batch_size, use_sae_substitution=False)
    full_logit_diff = mean_logit_diff(full_logits, dataset)

    all_features_mask = torch.ones(int(sae.cfg.d_sae), device=device)
    reconstruction_metrics = compute_faithfulness(
        all_features_mask,
        dataset,
        full_logit_diff=full_logit_diff,
        batch_size=batch_size,
    )
    print("All-feature SAE reconstruction:", reconstruction_metrics)

    rows = []
    for k in tqdm(k_values, desc="Magnitude top-K sweep"):
        mask = make_topk_mask(top_features, int(k))
        metrics = compute_faithfulness(
            mask,
            dataset,
            full_logit_diff=full_logit_diff,
            batch_size=batch_size,
        )
        rows.append({"k": int(k), **metrics})

    results = pd.DataFrame(rows)
    if output_csv_path is not None:
        output_csv_path = Path(output_csv_path)
        results.to_csv(output_csv_path, index=False)
        print(f"Saved results to {output_csv_path}")

    return results, reconstruction_metrics


def plot_pareto(results, reconstruction_metrics=None, output_path=None, title="Magnitude baseline Pareto curve"):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(results["k"], results["faithfulness"], marker="o", label="Magnitude top-K")
    ax.set_xscale("log")
    ax.set_xlabel("Circuit size (active SAE features)")
    ax.set_ylabel("Faithfulness (masked logit diff / full logit diff)")
    ax.set_title(title)
    ax.grid(True, which="both", alpha=0.3)
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=1.2, label="Full model")

    if reconstruction_metrics is not None:
        ax.axhline(
            reconstruction_metrics["faithfulness"],
            color="tab:orange",
            linestyle="--",
            linewidth=1.2,
            label="All-feature SAE reconstruction",
        )

    ax.legend()
    fig.tight_layout()
    assert ax.get_xscale() == "log", "Pareto x-axis must be log-scaled."

    if output_path is not None:
        output_path = Path(output_path)
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        print(f"Saved plot to {output_path}")

    plt.show()
    return fig, ax


## 9. Smoke test

This runs the whole pipeline on 8 prompts and `K=[5, 20]`. Run this before launching the 1,000-prompt experiment.


In [ ]:
smoke_mean_abs = cache_mean_abs_sae_features(
    smoke_dataset,
    cache_path=SMOKE_CACHE_PATH,
    batch_size=BATCH_SIZE,
    force_recompute=True,
)
smoke_top_features = smoke_mean_abs.argsort(descending=True)
smoke_results, smoke_reconstruction = run_magnitude_sweep(
    [5, 20],
    smoke_dataset,
    smoke_top_features,
    batch_size=BATCH_SIZE,
    output_csv_path=SMOKE_RESULTS_CSV_PATH,
)
write_run_manifest("smoke_completed", extra={"smoke_rows": len(smoke_results)})
smoke_results


In [ ]:
plot_pareto(
    smoke_results,
    reconstruction_metrics=smoke_reconstruction,
    output_path=SMOKE_PLOT_PATH,
    title="Smoke test: magnitude baseline",
)


## 10. Full MVP run

This is the intended MVP experiment: 1,000 generated IOI-style prompts, cached layer 8 SAE feature magnitudes, a top-K magnitude sweep, and a saved Pareto plot.


In [ ]:
mean_abs = cache_mean_abs_sae_features(
    full_dataset,
    cache_path=CACHE_PATH,
    batch_size=BATCH_SIZE,
    force_recompute=False,
)
top_features = mean_abs.argsort(descending=True)

print("Top 10 feature ids by mean absolute activation:")
print(top_features[:10].detach().cpu().tolist())


In [ ]:
results, reconstruction_metrics = run_magnitude_sweep(
    K_VALUES,
    full_dataset,
    top_features,
    batch_size=BATCH_SIZE,
    output_csv_path=RESULTS_CSV_PATH,
)
results


In [ ]:
plot_pareto(
    results,
    reconstruction_metrics=reconstruction_metrics,
    output_path=PLOT_PATH,
    title="Minimum SAE circuit discovery MVP: magnitude baseline",
)
manifest = write_run_manifest(
    "completed",
    extra={
        "rows": len(results),
        "reconstruction_metrics": reconstruction_metrics,
    },
)
copied = mirror_artifacts_to_latest([RESULTS_CSV_PATH, PLOT_PATH, MANIFEST_PATH])
print("Completed run manifest:")
print(json.dumps(manifest, indent=2))
print("Mirrored latest artifacts:")
for path in copied:
    print(path)


## 11. Expected artifacts

After the full run, Google Drive should contain this structure:

```text
MyDrive/minimum_sae_circuit_discovery/
  cache/v001_mvp_magnitude/
    mean_abs_sae_features_layer8_ioi.pt
    smoke_mean_abs_sae_features_layer8_ioi.pt
  runs/v001_mvp_magnitude/trial_YYYYMMDD_HHMMSS/
    run_manifest.json
    smoke_magnitude_pareto_results.csv
    smoke_magnitude_pareto.png
    magnitude_pareto_results.csv
    magnitude_pareto.png
  latest/v001_mvp_magnitude/
    run_manifest.json
    magnitude_pareto_results.csv
    magnitude_pareto.png
```

To create a new organized experiment version, change `RUN_VERSION`, for example to `v002_pso_search`. To force a manually named trial, set the `TRIAL_ID` environment variable before running the config cell; otherwise each run gets a timestamped trial folder.

Next experiments can reuse the cached feature magnitudes and replace the top-K mask selection with PSO, GA, SA, ACO, or upstream baselines.
